In [ ]:
import json
from pathlib import Path

# Find ATHAR project folder
project = Path.cwd()

if not (project / "data").exists():
    project = project.parent

# Dataset location
dataset_path = project / "data" / "raw" / "MultiSensor-Home1"

print("Dataset location:", dataset_path)

# Check required files
for filename in ["train.json", "test.json", "all_labels.json"]:
    path = dataset_path / filename
    print(filename, "FOUND" if path.exists() else "NOT FOUND")

In [ ]:
with open(dataset_path / "train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

print("Data type:", type(train_data).__name__)
print("Number of entries:", len(train_data))

if isinstance(train_data, list) and train_data:
    print("\nFirst entry:")
    print(str(train_data[0])[:2000])

elif isinstance(train_data, dict):
    print("\nTop-level keys:")
    print(list(train_data.keys())[:20])

    first_key = next(iter(train_data), None)
    if first_key is not None:
        print("\nFirst key:", first_key)
        print(str(train_data[first_key])[:2000])

In [ ]:
import json
from collections import Counter

# Load test data
with open(dataset_path / "test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("=== DATASET SUMMARY ===")
print("Training records:", len(train_data))
print("Testing records:", len(test_data))

# Count events and activity labels
activity_counts = Counter()
channels = Counter()
total_events = 0

for record in train_data:
    for event in record.get("tricks", []):
        total_events += 1
        activity_counts.update(event.get("labels", []))
        channels.update([event.get("channel")])

print("\nTotal annotated training events:", total_events)
print("Number of activity classes:", len(activity_counts))

print("\n=== ACTIVITY DISTRIBUTION ===")
for activity, count in activity_counts.most_common():
    print(f"{activity}: {count}")

print("\n=== CHANNEL VALUES ===")
print(dict(channels))

# Inspect the available label metadata
with open(dataset_path / "all_labels.json", "r", encoding="utf-8") as f:
    label_metadata = json.load(f)

print("\n=== ALL LABELS FILE ===")
print("Structure:", type(label_metadata).__name__)
print("Preview:", str(label_metadata)[:1200])

In [ ]:
with open(dataset_path / "all_labels.json", "r", encoding="utf-8") as f:
    all_data = json.load(f)

print("=== RECORD COUNTS ===")
print("Train:", len(train_data))
print("Test:", len(test_data))
print("All labels:", len(all_data))
print("Train + Test:", len(train_data) + len(test_data))

print("\n=== CAMERA NAMES ===")

for i in range(1, 6):
    key = f"video_url_{i}"
    print(f"Train camera {i}:", train_data[0][key])
    print(f"All camera {i}:  ", all_data[0][key])

print("\n=== FIRST RECORD EVENTS ===")
print(
    "Same annotations:",
    train_data[0]["tricks"] == all_data[0]["tricks"]
)

print("\n=== CHANNEL VALUES ===")
print(dict(channels))

In [ ]:
import re
from collections import Counter

# Normalize camera naming for comparison only
def record_key(record):
    path = record["video_url_1"]
    return re.sub(
        r"-(CAM\d+|View\d+)-",
        "-CAMERA-",
        path
    )

split_data = train_data + test_data

split_keys = [record_key(r) for r in split_data]
all_keys = [record_key(r) for r in all_data]

split_set = set(split_keys)
all_set = set(all_keys)

print("=== UNIQUE RECORDS ===")
print("Train + Test:", len(split_set))
print("All Labels:", len(all_set))

print("\n=== DUPLICATES ===")
print("Split duplicates:",
      sum(v - 1 for v in Counter(split_keys).values()))
print("All Labels duplicates:",
      sum(v - 1 for v in Counter(all_keys).values()))

print("\n=== RECORD DIFFERENCES ===")

extra = sorted(all_set - split_set)
missing = sorted(split_set - all_set)

print("Only in all_labels:", len(extra))
for item in extra[:20]:
    print(" EXTRA:", item)

print("Only in train/test:", len(missing))
for item in missing[:20]:
    print(" MISSING:", item)

# Compare annotations where keys are unique
split_lookup = {record_key(r): r for r in split_data}
all_lookup = {record_key(r): r for r in all_data}

if len(split_lookup) == len(split_keys) and len(all_lookup) == len(all_keys):
    mismatches = [
        key for key in (split_set & all_set)
        if split_lookup[key]["tricks"] != all_lookup[key]["tricks"]
    ]

    print("\n=== ANNOTATION CONSISTENCY ===")
    print("Matching record keys:", len(split_set & all_set))
    print("Annotation mismatches:", len(mismatches))
    print("Example mismatches:", mismatches[:5])
else:
    print("\nDuplicate keys detected: comparison needs review.")

In [ ]:
from pathlib import Path
import zipfile

base = Path(r"D:\Athar\data\raw\MultiSensor-Home1")

zip_path = base / "MultiSensor-Home1.zip"
output_dir = base / "sample_videos"

# First scenario: 5 camera views
sample_files = [
    f"01/00-View{i}-Part1.mp4"
    for i in range(1, 6)
]

with zipfile.ZipFile(zip_path, "r") as z:

    real_videos = [
        name for name in z.namelist()
        if name.lower().endswith(".mp4")
        and not name.startswith("__MACOSX/")
        and "/._" not in name
    ]

    print("Real MP4 files:", len(real_videos))

    for name in sample_files:

        if name not in z.namelist():
            print("MISSING:", name)
            continue

        destination = output_dir / name
        expected_size = z.getinfo(name).file_size

        if (not destination.exists()
                or destination.stat().st_size != expected_size):
            z.extract(name, output_dir)

        print(
            "READY:",
            name,
            round(destination.stat().st_size / (1024**2), 2),
            "MB"
        )

print("\nExtraction completed!")

In [ ]:
import cv2
from pathlib import Path

# Path to the extracted videos
video_dir = Path(
    r"D:\Athar\data\raw\MultiSensor-Home1\sample_videos\01"
)

print("=== ATHAR VIDEO INSPECTION ===\n")

for i in range(1, 6):

    video_path = video_dir / f"00-View{i}-Part1.mp4"

    if not video_path.exists():
        print(f"View{i}: FILE NOT FOUND")
        continue

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"View{i}: CANNOT OPEN VIDEO")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    duration = frames / fps if fps > 0 else 0

    print(f"Camera View {i}")
    print(f"  FPS: {fps:.2f}")
    print(f"  Total Frames: {frames}")
    print(f"  Resolution: {width} x {height}")
    print(f"  Duration: {duration:.2f} seconds")
    print("-" * 35)

    cap.release()

print("\nInspection completed!")


In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

video_dir = Path(
    r"D:\Athar\data\raw\MultiSensor-Home1\sample_videos\01"
)

timestamp = 4.5  # Seconds during Sitdown

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

for i in range(1, 6):
    video_path = video_dir / f"00-View{i}-Part1.mp4"

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Cannot open View{i}")
        axes[i-1].axis("off")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_number = int(timestamp * fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    success, frame = cap.read()

    if success:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        axes[i-1].imshow(frame_rgb)
        axes[i-1].set_title(f"Camera View {i}")
        axes[i-1].axis("off")
    else:
        print(f"Could not read View{i} at {timestamp}s")
        axes[i-1].axis("off")

    cap.release()

plt.suptitle(
    f"ATHAR - Multi-Camera Evidence at {timestamp}s",
    fontsize=14
)
plt.tight_layout()
plt.show()


In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

video_dir = Path(
    r"D:\Athar\data\raw\MultiSensor-Home1\sample_videos\01"
)

timestamps = [3.3, 4.5, 6.0]

fig, axes = plt.subplots(3, 5, figsize=(16, 10))

for row, t in enumerate(timestamps):
    for col in range(5):
        camera = col + 1
        path = video_dir / f"00-View{camera}-Part1.mp4"

        cap = cv2.VideoCapture(str(path))

        if not cap.isOpened():
            print("Cannot open:", path)
            axes[row, col].axis("off")
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_number = round(t * fps)

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(frame)
        else:
            print(f"Failed: View{camera} at {t}s")

        axes[row, col].set_title(f"View {camera} | {t}s")
        axes[row, col].axis("off")
        cap.release()

plt.suptitle("ATHAR - Sitdown Event Inspection")
plt.tight_layout()
plt.show()


In [ ]:

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

video_dir = Path(
    r"D:\Athar\data\raw\MultiSensor-Home1\sample_videos\01"
)

# ReadBook + Standup
timestamps = [10.0, 30.0, 55.0, 59.3, 60.0, 61.0]
cameras = [2, 3]

fig, axes = plt.subplots(
    len(timestamps), len(cameras),
    figsize=(9, 17)
)

for row, t in enumerate(timestamps):
    for col, camera in enumerate(cameras):

        path = video_dir / f"00-View{camera}-Part1.mp4"
        cap = cv2.VideoCapture(str(path))

        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.set(cv2.CAP_PROP_POS_FRAMES, round(t * fps))

        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(frame)
        else:
            print(f"Frame unavailable: View{camera}, {t}s")

        axes[row, col].set_title(f"View {camera} | {t}s")
        axes[row, col].axis("off")

        cap.release()

plt.suptitle("ATHAR - ReadBook & Standup Inspection")
plt.tight_layout()
plt.show()


In [ ]:

from IPython.display import Video, display
from pathlib import Path

video_dir = Path(
    r"D:\Athar\data\raw\MultiSensor-Home1\sample_videos\01"
)

for camera in [2, 3]:
    video_path = video_dir / f"00-View{camera}-Part1.mp4"

    print(f"Camera View {camera}")

    display(
        Video(
            filename=str(video_path),
            embed=True,
            width=550
        )
    )


In [ ]:

import csv
from pathlib import Path

project = Path(r"D:\Athar")
output = project / "data" / "annotations" / "home1_pilot_claims.csv"

claims = [
    {
        "claim_id": "HOME1_000_C01",
        "claim_text": "A person sits down on the sofa.",
        "source_event": "Sitdown",
        "start_s": 3.247,
        "end_s": 6.133,
        "camera_ids": "View2;View3",
        "preliminary_label": "SUPPORTED",
        "review_status": "PENDING"
    },
    {
        "claim_id": "HOME1_000_C02",
        "claim_text": "A person opens a book.",
        "source_event": "ReadBook",
        "start_s": "",
        "end_s": "",
        "camera_ids": "",
        "preliminary_label": "SUPPORTED",
        "review_status": "PENDING"
    },
    {
        "claim_id": "HOME1_000_C03",
        "claim_text": "A person reads a book.",
        "source_event": "ReadBook",
        "start_s": 7.524,
        "end_s": 59.073,
        "camera_ids": "",
        "preliminary_label": "",
        "review_status": "PENDING"
    },
    {
        "claim_id": "HOME1_000_C04",
        "claim_text": "A person stands up from the sofa.",
        "source_event": "Standup",
        "start_s": 59.123,
        "end_s": 61.104,
        "camera_ids": "View2;View3",
        "preliminary_label": "SUPPORTED",
        "review_status": "PENDING"
    }
]

output.parent.mkdir(parents=True, exist_ok=True)

with open(output, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=claims[0].keys())
    writer.writeheader()
    writer.writerows(claims)

print("Created:", output)
print("Total claims:", len(claims))


In [ ]:
import csv
from pathlib import Path
from collections import Counter

csv_path = Path(
    r"D:\Athar\data\annotations\home1_pilot_claims_v2.csv"
)

with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    claims = list(reader)

print("Total Claims:", len(claims))

print("\nLabel Distribution:")
print(dict(Counter(
    row["preliminary_label"] for row in claims
)))

print("\nValidation:")
for row in claims:
    valid = None not in row and all(
        value is not None for value in row.values()
    )
    print(row["claim_id"], "OK" if valid else "CHECK COLUMNS")


In [ ]:

import csv
from pathlib import Path

base = Path(r"D:\Athar\data\annotations")

source = base / "home1_pilot_claims_v2.csv"
output = base / "home1_reviewer2.csv"

with open(source, newline="", encoding="utf-8") as f:
    claims = list(csv.DictReader(f))

review_rows = []

for claim in claims:
    claim_id = claim["claim_id"]

    # Provide a time window for inspection.
    start = claim["evidence_start_s"] or claim["source_start_s"]
    end = claim["evidence_end_s"] or claim["source_end_s"]

    # C06 is specifically a claim about second 30.
    if claim_id == "HOME1_000_C06":
        start, end = "29.5", "30.5"

    review_rows.append({
        "claim_id": claim_id,
        "claim_text": claim["claim_text"],
        "review_start_s": start,
        "review_end_s": end,
        "available_camera_ids": claim["available_camera_ids"],
        "reviewer2_label": "",
        "reviewer2_evidence": "",
        "reviewer2_reason": ""
    })

with open(output, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=review_rows[0].keys()
    )
    writer.writeheader()
    writer.writerows(review_rows)

print("Created:", output)
print("Claims for review:", len(review_rows))
